# ERIKA, Part 1: Generate fictional records

Run this notebook once to create the local records used by **Part 2: Agent**. It uses only Python's standard library and does not connect to AWS. The dataset has 20 fictional patients, 47 visits, appointments, invoices, notes, and five teaching cases.

**Run order:** Run all cells top to bottom. The default output is `hospital_open_synthetic_data/` under the current notebook working directory. If you move the folder, set `DATA_DIR` in Part 2 to its new location. Reruns preserve existing payments and uploads.

## 1. Define the deterministic fixture

In [ ]:
"""Generate deterministic fictional hospital records for agent development only."""
from __future__ import annotations
import json
from datetime import date, timedelta
from pathlib import Path

CONDITIONS = [
    "hypertension", "type 2 diabetes", "asthma", "chronic kidney disease",
    "migraine", "osteoarthritis", "heart failure", "iron deficiency anemia",
]


def build_fixture() -> dict:
    patients, encounters, appointments, invoices, payments, documents = [], [], [], [], [], []
    for i in range(1, 21):
        pid = f"SYN-{i:03d}"
        condition = CONDITIONS[(i - 1) % len(CONDITIONS)]
        history = [condition, "prior fictional outpatient assessment"]
        patient = {
            "patient_id": pid, "name": f"Fictional Patient {i:03d}",
            "date_of_birth": f"19{75 + i % 20:02d}-04-15",
            "height_cm": 158 + i, "weight_kg": 56 + 2 * i,
            "history": history,
        }
        patients.append(patient)
        for j in range(1, 3):
            encounter_id = f"ENC-{i:03d}-{j}"
            start = date(2026, 8, 1) + timedelta(days=i + 12 * (j - 1))
            encounter = {
                "encounter_id": encounter_id, "patient_id": pid,
                "kind": "outpatient", "status": "completed",
                "visit_date": start.isoformat(), "admission_date": None,
                "discharge_date": None,
                "documented_diagnosis": condition,
                "treatment": f"SIM-PROTOCOL-{i:02d}",
                "documented_dose": f"{i} simulation units (fictional; not a real dose)",
                "charge_doctor": f"Dr. Demo {i % 4 + 1}",
                "charge_nurse": f"Nurse Demo {i % 5 + 1}",
            }
            encounters.append(encounter)
        if i <= 7:
            encounter_id = f"ENC-{i:03d}-3"
            start = date(2026, 9, 1) + timedelta(days=i)
            is_current = i == 7
            encounters.append({
                "encounter_id": encounter_id, "patient_id": pid,
                "kind": "inpatient", "status": "in_progress" if is_current else "completed",
                "visit_date": start.isoformat(), "admission_date": start.isoformat(),
                "discharge_date": None if is_current else (start + timedelta(days=3)).isoformat(),
                "documented_diagnosis": condition,
                "treatment": f"SIM-INPATIENT-PROTOCOL-{i:02d}",
                "documented_dose": f"{i} simulation units (fictional; not a real dose)",
                "charge_doctor": f"Dr. Demo {i % 4 + 1}",
                "charge_nurse": f"Nurse Demo {i % 5 + 1}",
            })
        invoice_cents = 5_000_000 if i == 1 else (30_000 + i * 5_000)
        invoices.append({"invoice_id": f"INV-{i:03d}", "patient_id": pid,
                         "total_cents": invoice_cents, "currency": "CAD"})
        payments.append({"payment_id": f"PAY-{i:03d}-1", "patient_id": pid,
                         "amount_cents": 1_000_000 if i == 1 else 10_000,
                         "recorded_date": "2026-09-15", "source": "seeded synthetic payment"})

    for encounter in encounters:
        patient = patients[int(encounter["patient_id"].split("-")[1]) - 1]
        doc_id = f"DOC-{encounter['encounter_id']}"
        text = (
            "SYNTHETIC TEST NOTE — NOT A REAL PATIENT RECORD\n"
            f"Patient: {patient['name']} ({patient['patient_id']})\n"
            f"Encounter: {encounter['encounter_id']} on {encounter['visit_date']}\n"
            f"Type: {encounter['kind']}; status: {encounter['status']}\n"
            f"Admission: {encounter['admission_date'] or 'not applicable'}\n"
            f"Discharge: {encounter['discharge_date'] or 'not documented'}\n"
            f"Documented diagnosis: {encounter['documented_diagnosis']}\n"
            f"History: {', '.join(patient['history'])}\n"
            f"Documented treatment: {encounter['treatment']}; "
            f"dose: {encounter['documented_dose']}\n"
            f"Charge doctor: {encounter['charge_doctor']}; "
            f"charge nurse: {encounter['charge_nurse']}\n"
        )
        documents.append({"document_id": doc_id, "patient_id": patient["patient_id"],
                          "encounter_id": encounter["encounter_id"], "file_name": f"{doc_id}.txt",
                          "source_type": "generated_note", "text": text})

    for i in range(12):
        appointments.append({
            "slot_id": f"SLOT-{i + 1:03d}",
            "doctor": f"Dr. Demo {i % 4 + 1}",
            "date": (date(2026, 10, 1) + timedelta(days=i // 2)).isoformat(),
            "time": "09:00" if i % 2 == 0 else "14:00",
            "status": "booked" if i in (1, 5, 9) else "available",
        })
    return {"synthetic": True, "version": 2, "patients": patients,
            "encounters": encounters, "appointments": appointments,
            "invoices": invoices, "payments": payments, "documents": documents}

## 2. Define the file writer and record store

The writer creates the fixture on the first run. The store reads it back for validation and can also ingest a sample fictional note.

In [ ]:
"""Open-access file tools for generated fictional hospital records only."""
from __future__ import annotations
import json
import uuid
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path


def _save_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)


def generate_synthetic_files(folder: str | Path) -> dict:
    """Create 20 fictional patient JSON records and 47 separate note files once."""
    root = Path(folder).expanduser().resolve()
    manifest_path = root / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("synthetic") is True and manifest.get("version") == 4:
            return manifest  # Reruns preserve simulated payments and uploads.
        raise ValueError("This folder contains another fixture version")
    if root.exists() and any(root.iterdir()):
        raise ValueError("Choose an empty folder for first generation")
    fixture = build_fixture()
    root.mkdir(parents=True, exist_ok=True)
    for person in fixture["patients"]:
        pid = person["patient_id"]
        docs = []
        for document in fixture["documents"]:
            if document["patient_id"] != pid:
                continue
            relative = f"documents/{pid}/{document['document_id']}.txt"
            destination = root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_text(document["text"], encoding="utf-8")
            docs.append({k: v for k, v in document.items() if k != "text"} | {"path": relative})
        record = {"synthetic": True, "patient": person,
                  "encounters": [e for e in fixture["encounters"] if e["patient_id"] == pid],
                  "documents": docs,
                  "invoice": next(i for i in fixture["invoices"] if i["patient_id"] == pid),
                  "payments": [p for p in fixture["payments"] if p["patient_id"] == pid]}
        _save_json(root / "patients" / f"{pid}.json", record)
    _save_json(root / "appointments.json", {"slots": fixture["appointments"]})
    manifest = {"synthetic": True, "version": 4,
                "patient_ids": [p["patient_id"] for p in fixture["patients"]],
                "generated_notes": len(fixture["documents"])}
    _save_json(manifest_path, manifest)
    return manifest


class OpenSyntheticStore:
    """No roles or patient isolation. Every fictional record is queryable."""
    def __init__(self, folder: str | Path):
        self.root = Path(folder).expanduser().resolve()
        self.manifest = json.loads((self.root / "manifest.json").read_text(encoding="utf-8"))
        if not (self.manifest.get("synthetic") is True and self.manifest.get("version") == 4):
            raise ValueError("Expected version 4 fictional files")
        self.patient_ids = tuple(self.manifest["patient_ids"])

    def _record(self, patient_id: str) -> dict | None:
        if patient_id not in self.patient_ids:
            return None
        return json.loads((self.root / "patients" / f"{patient_id}.json").read_text(encoding="utf-8"))

    def _records(self):
        for pid in self.patient_ids:
            yield self._record(pid)

    def hospital_stats(self) -> dict:
        encounters = [e for record in self._records() for e in record["encounters"]]
        inpatient = [e for e in encounters if e["kind"] == "inpatient"]
        return {"hospital_visits": len(encounters), "unique_patients": len(self.patient_ids),
                "admissions": len(inpatient),
                "discharges": sum(e["discharge_date"] is not None for e in inpatient),
                "currently_admitted": sum(e["discharge_date"] is None for e in inpatient)}

    def list_patient_names(self, group: str = "all") -> list[dict]:
        if group not in {"all", "admitted", "discharged", "currently_admitted"}:
            raise ValueError("group must be all/admitted/discharged/currently_admitted")
        result = []
        for r in self._records():
            admissions = [e for e in r["encounters"] if e["kind"] == "inpatient"]
            if group == "admitted" and not admissions: continue
            if group == "discharged" and not any(e["discharge_date"] for e in admissions): continue
            if group == "currently_admitted" and not any(e["discharge_date"] is None for e in admissions): continue
            result.append({k: r["patient"][k] for k in ("patient_id", "name")})
        return result

    def patient_record(self, patient_id: str) -> dict:
        r = self._record(patient_id)
        return {k: r[k] for k in ("patient", "encounters", "documents")} if r else {"error": "Patient not found"}

    def list_documents(self, patient_id: str = "") -> list[dict]:
        return [{k: d[k] for k in ("document_id", "patient_id", "encounter_id", "file_name", "source_type")}
                for r in self._records() if not patient_id or r["patient"]["patient_id"] == patient_id
                for d in r["documents"]]

    def _text(self, document: dict) -> str:
        path = (self.root / document["path"]).resolve()
        if not path.is_relative_to(self.root / "documents"):
            raise ValueError("Invalid document metadata path")
        return path.read_text(encoding="utf-8")

    def read_document(self, document_id: str) -> dict:
        for r in self._records():
            for d in r["documents"]:
                if d["document_id"] == document_id:
                    return {k: d[k] for k in ("document_id", "patient_id", "encounter_id", "file_name", "source_type")} | {"text": self._text(d)}
        return {"error": "Document not found"}

    def search_documents(self, query: str, patient_id: str = "") -> list[dict]:
        if not query.strip(): raise ValueError("Enter a search phrase")
        matches = []
        for d in self.list_documents(patient_id):
            text = self.read_document(d["document_id"])["text"]
            if query.casefold() in text.casefold():
                matches.append(d | {"preview": text[:250]})
        return matches

    def available_appointments(self, doctor: str = "", date: str = "") -> list[dict]:
        slots = json.loads((self.root / "appointments.json").read_text(encoding="utf-8"))["slots"]
        return [{k: slot[k] for k in ("slot_id", "doctor", "date", "time")}
                for slot in slots if slot["status"] == "available"
                and (not doctor or slot["doctor"] == doctor)
                and (not date or slot["date"] == date)]

    def calculate_bmi(self, patient_id: str) -> dict:
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        h, w = r["patient"]["height_cm"], r["patient"]["weight_kg"]
        return {"patient_id": patient_id, "height_cm": h, "weight_kg": w,
                "bmi": round(w / (h / 100) ** 2, 1), "formula": "weight_kg / (height_cm / 100)^2"}

    def billing_balance(self, patient_id: str) -> dict:
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        total = r["invoice"]["total_cents"]
        paid = sum(p["amount_cents"] for p in r["payments"])
        return {"patient_id": patient_id, "currency": "CAD", "invoice_id": r["invoice"]["invoice_id"],
                "total_cents": total, "paid_cents": paid, "balance_cents": max(0, total - paid),
                "credit_cents": max(0, paid - total), "payments": r["payments"]}

    def post_synthetic_payment(self, patient_id: str, amount_cad: str, receipt_id: str) -> dict:
        """Change fictional ledger only; repeat receipt IDs are idempotent."""
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        try: value = Decimal(amount_cad)
        except InvalidOperation as exc: raise ValueError("Invalid amount") from exc
        cents = value * 100
        if not value.is_finite() or cents <= 0 or cents != cents.to_integral_value():
            raise ValueError("Use a positive CAD amount with up to two decimals")
        if not receipt_id or len(receipt_id) > 80: raise ValueError("Receipt ID required")
        for other in self._records():
            for payment in other["payments"]:
                if payment["payment_id"] == receipt_id:
                    if payment["patient_id"] != patient_id or payment["amount_cents"] != int(cents):
                        raise ValueError("Receipt already used for different details")
                    return self.billing_balance(patient_id)
        r["payments"].append({"payment_id": receipt_id, "patient_id": patient_id,
            "amount_cents": int(cents), "recorded_date": datetime.now(timezone.utc).date().isoformat(),
            "source": "synthetic model test payment"})
        _save_json(self.root / "patients" / f"{patient_id}.json", r)
        return self.billing_balance(patient_id)

    def ingest_local_document(self, patient_id: str, local_path: str | Path) -> dict:
        """Notebook-side upload simulation; .txt/.md/.pdf/.docx, not a model tool."""
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        source = Path(local_path)
        if not source.is_file() or source.stat().st_size > 2_000_000:
            raise ValueError("File missing or larger than 2 MB")
        if source.suffix.lower() in {".txt", ".md"}:
            text = source.read_text(encoding="utf-8")
        elif source.suffix.lower() == ".pdf":
            from pypdf import PdfReader
            text = "\n".join(page.extract_text() or "" for page in PdfReader(source).pages[:50])
        elif source.suffix.lower() == ".docx":
            from docx import Document
            text = "\n".join(p.text for p in Document(source).paragraphs)
        else: raise ValueError("Supported types: .txt, .md, .pdf, .docx")
        if not text.strip() or len(text) > 200_000:
            raise ValueError("No extractable text or too long")
        doc_id = "UPLOAD-" + uuid.uuid4().hex[:12]
        relative = f"documents/{patient_id}/{doc_id}.txt"
        dest = self.root / relative
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(text, encoding="utf-8")
        r["documents"].append({"document_id": doc_id, "patient_id": patient_id,
            "encounter_id": None, "file_name": source.name, "source_type": "local_upload", "path": relative})
        _save_json(self.root / "patients" / f"{patient_id}.json", r)
        return {"patient_id": patient_id, "document_id": doc_id, "characters": len(text)}

## 3. Generate or reuse the data folder

Change `DATA_DIR` below if needed. A nonempty folder with another schema will be rejected.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "hospital_open_synthetic_data"
# Windows example: DATA_DIR = Path(r"C:\Users\YourName\Documents\hospital_open_synthetic_data")
manifest = generate_synthetic_files(DATA_DIR)
store = OpenSyntheticStore(DATA_DIR)
print("LOCAL DATA FOLDER:", DATA_DIR.resolve())
print("PATIENT FILES:", len(list((DATA_DIR / "patients").glob("SYN-*.json"))))
print("GENERATED NOTES:", manifest["generated_notes"])
print("COUNTS:", store.hospital_stats())

## 4. Validate generated records and notes

In [ ]:
balance = store.billing_balance("SYN-001")
print("Actual balance_cents:", balance["balance_cents"])
print("Invoice cents:", balance["total_cents"], "Paid cents:", balance["paid_cents"])
print("Seed balance was 4,000,000 cents; each successful test payment reduces it.")
assert balance["balance_cents"] == max(0, balance["total_cents"] - balance["paid_cents"])

In [ ]:
assert store.hospital_stats() == {"hospital_visits": 47, "unique_patients": 20,
    "admissions": 7, "discharges": 6, "currently_admitted": 1}
assert len(store.list_patient_names()) == 20
assert len(store.list_documents()) >= 47
for doc in store.list_documents():
    assert store.read_document(doc["document_id"])["patient_id"] == doc["patient_id"]
assert store.read_document("DOC-ENC-001-1")["patient_id"] == "SYN-001"
assert store.read_document("DOC-ENC-020-2")["patient_id"] == "SYN-020"
assert store.calculate_bmi("SYN-001")["bmi"] == 22.9
assert store.available_appointments(date="2026-10-01")[0]["slot_id"] == "SLOT-001"
# The fixture is persistent: later synthetic payment tests change this balance.
balance = store.billing_balance("SYN-001")
assert balance["total_cents"] == 5_000_000
assert any(p["payment_id"] == "PAY-001-1" and p["amount_cents"] == 1_000_000
           for p in balance["payments"])
assert balance["paid_cents"] == sum(p["amount_cents"] for p in balance["payments"])
assert balance["balance_cents"] == max(0, balance["total_cents"] - balance["paid_cents"])
assert balance["credit_cents"] == max(0, balance["paid_cents"] - balance["total_cents"])
print("PASS: all patients, every note, counts, appointments, BMI, balance, files")

## 5. Add a sample fictional upload

This optional demonstration writes a short text note once and links it to `SYN-002`. Remove or skip this cell if you only want the seeded files.

In [ ]:
SAMPLE_UPLOAD = NOTEBOOK_DIR / "fictional_uploaded_note.txt"
if not SAMPLE_UPLOAD.exists():
    SAMPLE_UPLOAD.write_text("FICTIONAL UPLOAD: Ask about the next follow-up visit.", encoding="utf-8")
existing = [d for d in store.list_documents("SYN-002") if d["file_name"] == SAMPLE_UPLOAD.name]
if not existing:
    print("INGESTED:", store.ingest_local_document("SYN-002", SAMPLE_UPLOAD))
else:
    print("REUSED UPLOAD:", existing[0]["document_id"])
print("UPLOAD SEARCH:", store.search_documents("next follow-up", "SYN-002"))
# To upload your own fictional file, change the path and patient ID below:
# print(store.ingest_local_document("SYN-005", r"C:\path\to\fictional_note.pdf"))

## 6. Create the teaching-case file

Part 2 reads this file; it does not create it.

In [ ]:
CASE_PATH = DATA_DIR / "fictional_training_cases.json"
if not CASE_PATH.exists():
    CASE_PATH.write_text('{\n  "synthetic": true,\n  "cases": [\n    {"case_id":"CASE-001","title":"Breathlessness during activity","presentation":"Fictional adult reports progressive breathlessness on exertion and ankle swelling over two weeks.","recorded_facts":"Known documented heart failure; current medicines and measurements not supplied.","teaching_focus":"Distinguish known diagnosis from current symptoms; identify missing severity and safety assessment."},\n    {"case_id":"CASE-002","title":"Single kidney result","presentation":"Fictional adult has one reported eGFR value and asks whether this proves chronic kidney disease.","recorded_facts":"No previous kidney results, urine findings or timeline supplied.","teaching_focus":"Identify missing chronicity evidence and avoid a definitive new diagnosis."},\n    {"case_id":"CASE-003","title":"Wheeze and shortness of breath","presentation":"Fictional adult reports wheezing and intermittent shortness of breath; history of asthma is mentioned.","recorded_facts":"No examination, current severity, oxygen saturation, or rescue plan documented.","teaching_focus":"Separate historical asthma from present assessment; identify escalation needs."},\n    {"case_id":"CASE-004","title":"Thirst and frequent urination","presentation":"Fictional adult reports thirst and frequent urination and asks whether this means type 2 diabetes.","recorded_facts":"No glucose measurement or prior diagnosis supplied.","teaching_focus":"Avoid diagnosing from symptoms alone; ask what data and assessment are needed."},\n    {"case_id":"CASE-005","title":"Sudden severe headache","presentation":"Fictional adult reports a sudden severe headache and new weakness of one arm.","recorded_facts":"Onset is sudden; no examination or imaging available.","teaching_focus":"Treat this as an urgent concern, avoid a specific diagnosis, and advise immediate emergency assessment."}\n  ]\n}\n', encoding="utf-8")

## Output for Part 2

The folder contains `manifest.json`, `appointments.json`, `patients/SYN-*.json`, `documents/SYN-*/...txt`, and `fictional_training_cases.json`. Copy the folder to the machine where you run Part 2, or point Part 2 at this folder. Do not use real patient information in this educational fixture.